In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 21
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## Powell-Thyne Coup Pipeline

**Source:** Powell & Thyne Coups Database
**Access:** Automated direct TXT download — no registration required
**Download instructions:** See `docs/instructions_data_maintenance.md` — POWELL_THYNE section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Coup attempt (any) | Political settlement | Supplementary |
| Successful coup | Political stability | Primary tier 1 |
| Failed coup | Political stability | Primary tier 1 |

In [3]:
import requests
import io
import pandas as pd
from datetime import datetime

# Powell-Thyne country-year coup data — direct TXT download
# Updated continuously; version date embedded in data
POWELL_THYNE_URL = "http://www.uky.edu/~clthyn2/coup_data/powell_thyne_ccode_year.txt"

print("Downloading Powell-Thyne coup data...")
response = requests.get(POWELL_THYNE_URL, timeout=60)
print(f"Status: {response.status_code}, Size: {len(response.content)/1024:.1f}KB")

pt_raw = pd.read_csv(io.StringIO(response.text), sep='\t')
print(f"\nShape: {pt_raw.shape}")
print(f"Columns: {list(pt_raw.columns)}")
print(f"Years: {pt_raw['year'].min()} — {pt_raw['year'].max()}")
print(f"Countries: {pt_raw['ccode'].nunique()}")
print(pt_raw.head(3))

Status: 200, Size: 691.7KB

Shape: (12384, 15)
Columns: ['ccode', 'abbrev', 'country', 'year', 'ccode_gw', 'ccode_polity', 'coup1', 'coup2', 'coup3', 'coup4', 'date1', 'date2', 'date3', 'date4', 'version']
Years: 1950 — 2025
Countries: 204
   ccode abbrev                   country  year  ccode_gw  ccode_polity  \
0      2    USA  United States of America  1950       NaN           NaN   
1      2    USA  United States of America  1951       NaN           NaN   
2      2    USA  United States of America  1952       NaN           NaN   

   coup1  coup2  coup3  coup4 date1 date2 date3 date4      version  
0      0      0      0      0   NaN   NaN   NaN   NaN  V2026.01.13  
1      0      0      0      0   NaN   NaN   NaN   NaN  V2026.01.13  
2      0      0      0      0   NaN   NaN   NaN   NaN  V2026.01.13  


In [4]:
# Filter to framework start year and select relevant columns
pt = pt_raw[['ccode', 'abbrev', 'country', 'year', 'coup1', 'coup2', 'coup3', 'coup4', 'version']].copy()

# Rename columns
pt = pt.rename(columns={
    'ccode':   'cow_code',
    'abbrev':  'country_code',
    'country': 'country_name',
    'coup1':   'pt_coup_successful',
    'coup2':   'pt_coup_failed',
    'coup3':   'pt_coup_alleged',
    'coup4':   'pt_autocoup',
    'version': 'pt_version',
})

# Derive data currency from version field — no hardcoding
latest_version = pt['pt_version'].dropna().iloc[0]
data_as_of_date = f"{latest_version[1:5]}-{latest_version[5:7]}"

# Filter to framework start year
pt = pt[pt['year'] >= FRAMEWORK_START_YEAR].copy()
pt = pt.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {pt.shape}")
print(f"Years: {pt['year'].min()} — {pt['year'].max()}")
print(f"Countries: {pt['country_name'].nunique()}")
print(f"Data version: {latest_version}")
print(f"\nCoup counts:")
print(f"  Successful coups: {pt['pt_coup_successful'].sum()}")
print(f"  Failed attempts:  {pt['pt_coup_failed'].sum()}")
print(f"  Alleged coups:    {pt['pt_coup_alleged'].sum()}")
print(f"  Auto-coups:       {pt['pt_autocoup'].sum()}")
print(f"\nMissing values: {pt.isnull().sum()[pt.isnull().sum() > 0].to_dict()}")

Shape: (6961, 9)
Years: 1990 — 2025
Countries: 201
Data version: V2026.01.13

Coup counts:
  Successful coups: 174
  Failed attempts:  15
  Alleged coups:    1
  Auto-coups:       1

Missing values: {}


In [6]:
# Derive data currency from version string — format is V{YYYY}.{MM}.{DD}
latest_version = pt['pt_version'].dropna().iloc[0]
data_as_of_date = f"{latest_version[1:5]}-{latest_version[6:8]}"
print(f"Data as of: {data_as_of_date}")

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "powell_thyne_clean.csv")
pt.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {pt.shape}")

# Update download log
update_entry(
    "POWELL_THYNE",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="powell_thyne_clean.csv",
    latest_available_version=latest_version,
    notes="Country-year coup data. Variables: successful, failed, alleged, auto-coup. Coverage: 1950-2025, 204 countries. Version auto-detected from data."
)

print_entry("POWELL_THYNE")

Data as of: 2026-01
Written: /Users/boulanger/Documents/governance-framework/data/processed/powell_thyne_clean.csv
Shape: (6961, 9)
[download_log] Updated entry for POWELL_THYNE
  source_id: POWELL_THYNE
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-12
  data_as_of_date: 2026-01
  local_filename: powell_thyne_clean.csv
  latest_available_version: V2026.01.13
  no_update_reason: nan
  notes: Country-year coup data. Variables: successful, failed, alleged, auto-coup. Coverage: 1950-2025, 204 countries. Version auto-detected from data.
